# 1 · Data cleaning

Raw scrape (42,675 rows) → `amazon_products_deduped.csv` (one row per product, 7,946 rows; used by everything else), plus a listing-level file (one row per scrape). Reasons for each step: [`METHODOLOGY.md`](../METHODOLOGY.md).

In [1]:
import re
import numpy as np
import pandas as pd

df = pd.read_csv('../data/amazon_sales_data_uncleaned.csv')
print(df.shape)

(42675, 16)


## Parse raw fields

Text → numbers with regex; missing prices left missing (99% out of stock); columns nothing downstream uses are dropped.

In [2]:
before = len(df)
df = df.drop_duplicates()
print(f'dropped {before - len(df)} duplicate rows')

df['rating'] = df['Rating'].str.extract(r'([\d.]+)').astype(float)
df['number_of_reviews'] = df['Number_of_reviews'].str.replace(',', '', regex=False).astype(float)
df = df.drop(columns=['Rating', 'Number_of_reviews'])

def parse_bought(val):
    if not isinstance(val, str):
        return np.nan
    m = re.match(r'([\d.]+)(K)?\+', val)
    if not m:
        return np.nan
    num = float(m.group(1))
    if m.group(2) == 'K':
        num *= 1000
    return num

df['bought_in_last_month'] = df['bought_in_last_month'].apply(parse_bought)

def to_price(series):
    return pd.to_numeric(
        series.astype(str).str.replace('$', '', regex=False).str.replace(',', '', regex=False),
        errors='coerce',
    )

df['current_price'] = to_price(df['Current/discounted_price'])  # shelf price; coupons are temporary
df['is_sponsored'] = df['is_sponsored'] == 'Sponsored'
df['in_stock'] = df['buy_box_availability'] == 'Add to cart'
# unused downstream: list/variant price, coupons, badges, delivery text, URLs, images
df = df.drop(columns=['Current/discounted_price', 'Listed_price', 'Price_on_variant', 'is_couponed', 'is_best_seller',
                      'buy_box_availability', 'sustainability_badges', 'delivery_details', 'product_url', 'image_url'])
df['collected_at'] = pd.to_datetime(df['collected_at'], format='%d-%m-%Y %H:%M')

no_price = df['current_price'].isna()
print(f"current_price missing on {no_price.mean():.0%} of rows, {(~df.loc[no_price, 'in_stock']).mean():.0%} of them out of stock")
df[['rating', 'number_of_reviews', 'bought_in_last_month', 'current_price']].describe().round(1)

dropped 962 duplicate rows
current_price missing on 28% of rows, 99% of them out of stock


,rating,number_of_reviews,bought_in_last_month,current_price
count,40689.0,40689.0,31202.0,30125.0
mean,4.4,3046.5,944.9,217.8
std,0.4,13100.1,4941.2,393.7
min,1.0,1.0,50.0,2.5
25%,4.2,77.0,100.0,29.7
50%,4.5,340.0,200.0,80.0
75%,4.6,1845.0,400.0,200.0
max,5.0,865598.0,100000.0,4699.0


## Sales bins

In stock with no sales badge → 0 (Amazon hides it below 50); out of stock → missing.

In [3]:
below_50 = df['bought_in_last_month'].isna() & df['in_stock']
df.loc[below_50, 'bought_in_last_month'] = 0
print(f"{below_50.sum()} in-stock listing rows set to 0 (<50); "
      f"{df['bought_in_last_month'].isna().sum()} out-of-stock rows left missing")

4520 in-stock listing rows set to 0 (<50); 5991 out-of-stock rows left missing


## Category: keyword rules

A category is what the product *is*. Rules match the title text before "for / compatible with" first, then the full title.

In [4]:
# Bags and hard cases made for electronics: a laptop backpack is an accessory,
# not a laptop, and would otherwise be filed by its device word (Computers,
# Cameras). Requires a device word somewhere in the title, and the bag noun
# before any "with"/"bundle" (so "Camera Bundle with Bag" stays a camera).
# Generic bags with no device word fall through to Other -> Non-Electronics.
BAGS = (
    r"^(?!.*(airtag|otterbox|grab bag))"
    r"(?=.*\b(laptops?|notebooks?|computers?|macbook|chromebook|tablets?|ipad|cameras?|dslr|slr|mirrorless|drones?|electronics|tech)\b)"
    r"(?:(?! with | w/ |bundle| \+ ).)*?(backpack|\bbag\b|\bsleeve\b|laptop case|briefcase|\btote\b|sling bag|everyday sling|messenger bag)"
    r"|pelican (air )?\d{4}|nanuk|electronics organizer"
)
# Phones and tablets themselves (incl. renewed), not their accessories: the
# title starts with the device name (after brand / "Renewed" / year words) and
# has no accessory noun. Checked early: tablet titles mention "RAM"/"Display".
PHONES = (
    r"^(?!.*\b(cases?|covers?|protectors?|chargers?|cables?|stands?|holders?|mounts?|stylus|keyboard|folio|skins?|straps?|bands?"
    r"|adapters?|docks?|grips?|lens|tripod|gimbal|earbuds|headphones|headsets?|expansion|wallet|sleeve|bag|replacement|parts?|tool|s[- ]?pen for)\b)"
    r"(?:(?:renewed|new|\(renewed\)|\d{4}|apple|samsung|google|motorola|oneplus|amazon|lenovo|tcl|nokia)\W+)*"
    r"(?:iphone|ipad|galaxy (?:s\d|a\d|z ?(?:fold|flip)|tab|note)\w*|pixel \d|kindle|moto [gez]|oneplus (?:pad|\d+|nord)|fire (?:hd )?\d+\b|tab [a-z]?\d)"
    r"|\b(?:unlocked smartphone|android tablet|cordless phone|dect 6\.0|corded phone|voip phone|ip phone|desk phone|landline)\b"
    r"|(?:corded|cordless)[\w/ -]{0,20}?\b(?:tele)?phone\b|corded answering system"  # "Corded/Cordless Senior Phone", "Corded Telephone"
)
# Smartwatches / fitness bands whose titles would otherwise hit "galaxy" or
# "heart monitor" first. Bands and chargers for them are excluded.
WATCHES = r"^(?!.*\b(bands?|straps?|chargers?|cases?|protectors?|charging)\b).*(galaxy (?:watch|fit|ring)|\bsmartwatch\b|smart watch)"
# SSDs are storage whether internal or external (like hard drives); laptops
# and PCs that list an SSD spec are excluded (they still reach Computers via \bssd\b).
SSD = r"^(?!.*\b(laptop|notebook|ultrabook|desktop|mini pc|gaming pc|computer|chromebook|chromebox|macbook|mac mini|mac studio|surface|thinkpad|workstation|tablet|tab|ipad|console|handheld|ram|cpu|processor|intel|ryzen|celeron|windows|motherboard|mini-itx|microatx|heatsink|bracket|hub)\b).*(\bssd\b|solid state drive|\bnvme\b)"
# Video production / conferencing gear goes with cameras (user's call).
VIDEO = r"webcam|\batem\b|blackmagic|rally (?:bar|plus)|video ?conferenc|conference (camera|room)|meetingbar|\bmeetup\b|capture card|cam link|game capture|\belgato hd60\b"
# UPS battery backups are surge protectors with a battery (user's call, as in the
# guidelines); their replacement battery cartridges are batteries. Checked early:
# UPS titles also say "battery", "USB charging ports", "TV & computer".
UPS_BATTERY = r"\bups\b.*\b(replacement battery|battery replacement|battery cartridge)\b|\b(replacement battery|battery replacement)\b.*\bups\b"
UPS = r"^(?!.*\b(label printer|rails?|calculator)\b).*(\bups\b|uninterruptible)"
SMART_LOCK = r"smart lock|smart entry|smart module|(deadbolt|door lock).*(wi-?fi|keypad|fingerprint)|(wi-?fi|keypad|fingerprint).*(deadbolt|door lock)"
# Conference-room video systems also list microphones/speakers/PC compatibility.
VIDEO_SYSTEM = r"\bmeetup\b|rally (?:bar|plus)|video conferencing system|conference room camera|video bar"
# Blank camera media: "Compact Flash Card" otherwise hits the camera "flash"
# keyword, "Video Cassette" the audio "cassette" keyword, "SDHC Trail Camera
# Flash Memory" the camera "trail cam" keyword.
CAMERA_MEDIA = r"video cassette|\bminidv\b|compact ?flash|\bcf card|^(?!.*bundle).*\b(sdhc|sdxc)\b"
# Household ventilation fans; one title also says "Installation Kit" (a car keyword).
HOME_FAN = r"exhaust fan|ventilation fan"
# "Trilingual Display" otherwise sends a weather radio to Monitors.
WEATHER_RADIO = r"weather (alert )?radio"
CATEGORY_PATTERNS = [
    ("Bags & Cases", BAGS),
    ("Phones & Tablets", PHONES),
    ("Smart Home & Wearables", WATCHES),
    ("Storage & Memory Cards", SSD),
    ("Batteries", UPS_BATTERY),
    ("Power Strips & Surge Protectors", UPS),
    ("Smart Home & Wearables", SMART_LOCK),
    ("Cameras & Photography", VIDEO_SYSTEM),
    ("Storage & Memory Cards", CAMERA_MEDIA),
    ("Non-Electronics", HOME_FAN),
    ("Audio", WEATHER_RADIO),
    ("Protection Plan", r"asurion|protection plan|complete protect|\d+ year (home|major appliance)"),
    ("Printers & Ink", r"\bprinter\b|\bink\b|toner|cartridge|laserjet|\bmfp\b|photo paper|presentation paper|project paper|business paper|matte paper|bond paper|printhead|drum unit|\bscanjet\b|flatbed|document scanner|photo scanner|(?<!barcode )(?<!digital )(?<!radio )\bscanner\b|fax machine"),
    ("Office & School Supplies", r"\blabels?\b|\bavery\b|\bdymo\b|calculator|texas instruments|barcode scanner|mailing label|printable label|laminat|shredder|post-it|sticky note|binding (machine|comb)"
                                 r"|file folders?|hanging (file )?folders?|folder|\bstapler\b|\bstaples\b|\bpencils\b|(mechanical|drafting|#2) pencil|envelopes?|binder clips|dry[- ]erase|whiteboard|copy paper|index cards|card ?stock|sheet protector|tape dispenser|packing tape|\bmarkers\b|clipboard|desk organizer|lined notebook|note pads|\bpapers \|"),
    ("Computers & Laptops", r"laptop|desktop|macbook|chromebook|\bssd\b|\bram\b|motherboard|\bcpu\b|processor|\bpcie\b|\bnvme\b|docking station|(?<!charging )(?<!mouse )\bdock\b|cooling fan|\bpc fans?\b|\bgpu\b|graphics (card|board)|\brtx\b|\bgtx\b"
                            r"|ryzen|\bcore i[3579]\b|\bi[3579]-\d{4,5}|\batx\b|80 ?\+|80 plus|modular power supply|\d{3,4}w\b.*power supply|\bpsu\b|argb fan|pwm (reverse )?fans?|fan hub|\d{3}mm fan|cpu cooler|liquid cooler|sodimm|\bddr[345]\b|\bquadro\b"
                            r"|computer case|mid-tower|pc case|\bjetson\b|arduino|raspberry pi|microcontroller|breadboard|developer kit"),
    ("Smart Home & Wearables", r"digital (picture|photo) frame|(picture|photo) frame.*wi-?fi|wi-?fi.*(picture|photo) frame"),  # user's call
    ("Monitors & Displays", r"(?<!baby )(?<!rate )(?<!reference )(?<!tv )\bmonitor\b|(?<!lcd )(?<!led )(?<!oled )(?<!backlit )(?<!color )(?<!touch )\bdisplay\b(?!\s?port)"),
    # Checked before Audio: mouse/keyboard titles mention a "USB receiver",
    # webcams advertise "stereo audio", and power banks mention iPhone/battery,
    # all of which match keywords in categories checked earlier.
    # tablet keyboard cases are tablet accessories (Surface covers stay with Computers, user's call)
    ("Phone & Tablet Accessories", r"^(?=.*\bkeyboard\b)(?=.*\b(case|folio|cover)\b)(?=.*\b(ipad|tablet|galaxy tab|fire hd|fire max)\b)"),
    ("Keyboards & Mice", r"\bkeyboard\b(?!\s*(case|folio|cover))|\bmouse\b|mouse ?pad|trackpad|wrist rest|palm rest|stream deck|unifying receiver"),
    ("Cameras & Photography", VIDEO),
    ("Cables & Charging", r"power bank|portable charger"),
    ("Smart Home & Wearables", r"wake-up light"),
    ("TV & Home Entertainment", r"projector(?! mount)"),
    ("Marine & GPS", r"fish ?finder|transducer|chartplotter|depth sounder|marine radio|humminbird|lowrance|satellite communicator|\binreach\b"),
    ("Car Electronics", r"radar detector|radar/laser|laser/radar|laser detection|double din|installation kit|wiring harness|dash ?cam|smartcord|car antenna|antenna compatible with"),
    ("Optics", r"binocular|telescope|monocular|microscope|eyepiece|spotting scope|observatory|rangefinder"),
    # Audio checked before Phone & Tablet Accessories: audio product titles very
    # commonly mention "for iPhone"/"Phone Calls" compatibility, which would
    # otherwise misclassify headphones/earbuds as phone accessories (verified:
    # 189/1,669 "headphone" titles, incl. AirPods/Sony WH-1000XM4, were wrongly
    # landing in Phone & Tablet Accessories before this reorder).
    ("Audio", r"speaker|headphone|headset|earbud|airpods|microphone|\bmic\b|sound ?bar|\bstereo\b|two-way radio|walkie[- ]talkie|turntable|av receiver|(?<!usb )(?<!nano )(?<!unifying )(?<!bolt )(?<!ghz )\breceiver\b|cb radio"
              r"|(?<!repeater )(?<!antenna )(?<!signal )\bamplifier|subwoofer|bass module|earphones|homepod|\bradio\b|voice recorder|mp3 player|\bdac\b|stylus \d|replacement stylus|record (brush|player)|vinyl|surround sound|cassette|packtalk|sustain pedal|\bsonos\b|\bcomponents\b"),
    ("Phone & Tablet Accessories", r"keyboard (case|folio|cover)|iphone|\bipad\b|galaxy|\bphone case\b|screen protector|phone mount|phone holder|\bphone\b|\btablet\b|\bkindle\b|e-?reader|\bpixel \d|airtag|popsocket|poptop|moto tag|apple pencil|stylus|active pen|tilt pen|\bpen\b"),
    ("Cameras & Photography", r"\bcamera\b|\bcam\b|\blens\b|\bcanon\b|\bnikon\b|gopro|tripod|monopod|\bdslr\b|fujifilm|\binstax\b|polaroid|\bfilm\b|\b135-\d{2}\b|\d+-\d+mm f/?\d|\d+mm f/?\d|\bf\d\.\d\b|\bdg dn\b|teleconverter"
                              r"|speedlite|\bflash\b(?!\s*(drive|memory))|strobe|softbox|light dome|diffus|bowens|key light|video light|ring light|tube light|selfie stick|backdrop|photo studio|bounce card|trail cam|capture card|video recorder|camcorder|\bdrone\b|gimbal|stabilizer"),
    ("Cables & Charging", r"hdmi (fiber|optic)"),
    ("Networking", r"\bwifi\b|wi-fi|router|range extender|ethernet|\bmodem\b|\bnetgear\b|\bpoe\b|server rack|access point|\bunifi\b|\bomada\b|network switch|patch panel|keystone|\brj45\b|nanobeam|cloud gateway"),
    ("Gaming", r"\bxbox\b|playstation|\bps5\b|\bps4\b|nintendo|gaming controller|\bgamepad\b|(?<!with )(?<!rgb )(?<!remote )\bcontroller\b|joy-con|amiibo|meta quest|\bhosas\b|flight (pack|stick)|racing (cockpit|wheel)|simulator cockpit|arcade|joystick|game & watch|8bitdo"),
    ("TV & Home Entertainment", r"\btv\b|remote control|streaming device|\bfire tv\b|\broku\b|projector|projection screen|blu[- ]?ray (dvd |disc )?player|dvd player|\bhdtv antenna|indoor antenna|universal remote"),
    ("Storage & Memory Cards", r"memory card|microsd|flash drive|\bhard drive\b|external drive|\bsdhc\b|\bsdxc\b|memory flash|\bhdd\b|\bnas\b|diskstation|cfexpress|card reader|dvd[+-]r|cd-r|recordable|rewritable|blu-ray (drive|burner)|read/write|\bburner\b|optical drive|dvd/cd|dvd rw|bdxl|\bdiscs?\b"),
    ("Smart Home & Wearables", r"\balexa\b|\becho\b|smart plug|\bnest\b|\bring\b doorbell|doorbell|smart ?watch|smart band|fitness (tracker|band)|activity[- ]tracker|\bgps\b|cycling computer|\bhrm\b|watch band|band for apple watch|quickfit|\bfitbit\b"
                               r"|motion sensor|entry sensor|door sensor|window and door|security system|simplisafe|wake-up light|electronic safe|home safe"),
    ("Batteries", r"\bbatteries\b|\bbattery\b|\blithium\b.*\bcell\b|\benergizer\b|\bduracell\b|\bcell pack\b|3v lithium"),
    ("Power Strips & Surge Protectors", r"surge protector|power strip|extension cord|\boutlet\b|power conditioner|timer switch|wall ?plate|circuit breaker|surgebreaker|gfci|light switch|power distribution|\bpdu\b|surge protective"),
    ("Cables & Charging", r"\bcable\b|\bcharger\b|charging|\bhdmi\b|\busb-c\b|power adapter|power bank|\badapter\b|\bups\b|\bhub\b|cord (bundlers|organizer)|connectors?\b|crimp terminal|tie wire"),
]
# Not consumer electronics: bags, toys, personal care, household, kitchen,
# pet, furniture, hand tools, apparel. Only applied to titles that no category
# above claimed (i.e. would otherwise be "Other"), so a device title that
# merely mentions a backpack/dog/wallet is never caught by it.
NON_ELECTRONICS = re.compile(r"backpack|\btote\b|daypack|shoulder bag|travel pack|\bpuzzle|\btoys?\b|play food|pretend play|melissa & doug|crayola|oil pastel|\bgame \(\d+ pcs"
                        r"|deodorant|antiperspirant|mouthwash|hand ?wash|hand sanitizer|ointment|facial tissue|paper towels|car wash|car cleaning|cleaning wipes|screen wipes|screen cleaner|wipes for car|cleaning cloth|chamois|respirator"
                        r"|\bdogs?\b|\bpuppy\b|\bpet\b|bakeware|loaf pan|paring knife|pepper mill|toaster|kettle|hand mixer|multi-cooker|\bopener\b|coffee maker and espresso machine cleaner|cooktop"
                        r"|desk chair|task chair|mid-back|\bfoot rest\b|storage shed|wire rack|patio heater|humidifier|lantern"
                        r"|screwdriver|wire cutter|circular saw|staple gun|combination lock|deadlatch|heat shrink|velcro"
                        r"|guitar strings|guitar stand|ukulele|wristband|sweatbands|dri fit|shaver|\brazor\b|beard trimmer|haircut|hearing protector|soil color book|world's most amazing|play slide|remote starter|mobile wallet|golf", re.I)

# Anything sold for use in a car goes in Car Electronics (user's call: car
# chargers, car amps, FM transmitters, radio scanners are bought for the car).
# Applied after the keyword match, to titles that landed in a generic category.
CAR = re.compile(r"\bcar (audio|stereo|speakers?|amplifier|amp|charger|radio)\b|\bcar\b.*\bcharger\b|cigarette lighter|fm transmitter"
                 r"|\bscanner\b.*\b(police|trunk ?tracker|channels|bearcat|beartracker)\b|\b(police|bearcat|beartracker|uniden)\b.*\bscanner\b|radio scanner", re.I)
CAR_FROM = {"Audio", "Cables & Charging", "Printers & Ink", "Computers & Laptops", "Other"}  # Computers: "... Car Charger for ... MacBook"

# Plain stationery is not electronics (user's call, 2026-09-25): dropped with the
# non-electronics. Kept in Office & School Supplies: the devices (calculators,
# label makers, shredders, laminators, presenters), their supplies (label tapes,
# laminating pouches, binding combs) and printable paper/cards/labels, which are
# printing supplies like photo paper.
STATIONERY = re.compile(r"\bbinders?\b|binder clips|folders?\b|\bpencils?\b|\bpens?\b|\bmarkers?\b|sticky notes|post-it"
                        r"|note ?pads?|\bnotebook\b|writing pad|index cards|\bstaplers?\b|\bstaples\b|staple gun|whiteboard"
                        r"|white board|dry[- ]erase|clipboard|envelopes?|dividers|sheet protectors?|organizer|easel pad"
                        r"|coloring|name badges?|name tags?|tab inserts|spine inserts|expanding file", re.I)
OFFICE_DEVICE = re.compile(r"calculator|label ?maker|labelmanager|labelwriter|printer\b|laminat|shredder|presenter|electric"
                           r"|usb|bluetooth|battery|digital|wireless|barcode|scanner|air duster|compressed air|\bcd\b|\bdvd\b"
                           r"|binding (machine|comb)", re.I)
PRINT_SUPPLY = re.compile(r"printable|\blabels?\b|tape|pouch|p-touch|\btze\b|\bpaper\b|card ?stock", re.I)
ALWAYS_STATIONERY = re.compile(r"ring binders?|economy view|easel pad", re.I)

def is_stationery(title):
    if OFFICE_DEVICE.search(title):
        return False
    return bool(ALWAYS_STATIONERY.search(title)) or (bool(STATIONERY.search(title)) and not PRINT_SUPPLY.search(title))

COMPAT = re.compile(r'\s(?:for|compatible with|works with)\s', re.I)
DROP_CATEGORIES = {"Protection Plan", "Non-Electronics"}

def _match(text):
    t = text.lower()
    for cat, pattern in CATEGORY_PATTERNS:
        if re.search(pattern, t):
            return cat
    return "Other"

def categorize(title):
    head = COMPAT.split(title, maxsplit=1)[0]
    cat = _match(head) if len(head) >= 15 else "Other"
    cat = cat if cat != "Other" else _match(title)
    if cat in CAR_FROM and CAR.search(title):
        return "Car Electronics"
    if cat == "Office & School Supplies" and is_stationery(title):
        return "Non-Electronics"
    if cat == "Other" and NON_ELECTRONICS.search(title):
        return "Non-Electronics"
    return cat


df['category'] = df['Title'].apply(categorize)
df.drop_duplicates('Title')['category'].value_counts()

category
Computers & Laptops                1231
Audio                              1158
Printers & Ink                      910
Cameras & Photography               702
Protection Plan                     666
Phone & Tablet Accessories          428
Monitors & Displays                 376
Cables & Charging                   324
Batteries                           324
Networking                          313
Storage & Memory Cards              305
Keyboards & Mice                    286
Office & School Supplies            262
Smart Home & Wearables              206
Phones & Tablets                    185
TV & Home Entertainment             182
Non-Electronics                     171
Gaming                              160
Other                               151
Power Strips & Surge Protectors     134
Car Electronics                     130
Bags & Cases                        109
Marine & GPS                         49
Optics                               46
Name: count, dtype: int64

## Brand

First word of the title, after stripping trademark symbols and skipping quantities; generic first words → first known brand before "for", else `Unbranded`.

In [5]:
NON_BRAND_SHAPE = re.compile(
    r'^\(?-?\d+(-\d+)?\)?$|^\d+-?(gb|tb|mb|mm|xl|pk|pc|pcs|ct|cou|pack)$|^(x|count|pcs|pc|ct|new|the)$|^[^a-z0-9]+$',
    re.I,
)

# title-prefix pattern -> canonical brand, for brands the first word would truncate or misspell
MULTI_WORD_BRANDS = {
    r'amazon ?basics': 'Amazon Basics',  # Amazon's private label, distinct from Amazon devices (Echo, Kindle)
    r'texas instruments|ti[- ]?\d\d\w*': 'Texas Instruments',
    r'western digital': 'Western Digital',
    r'the north face': 'The North Face',
    r'cooler ?master': 'Cooler Master',
    r'lian[- ]li': 'Lian Li',
    r'nite ize': 'Nite Ize',
    r'mounting dream': 'Mounting Dream',
    r'insta ?360': 'Insta360',
    r'audio[- ]technica': 'Audio-Technica',
    r'vic firth': 'Vic Firth',
    r'sound ?blaster\w*': 'Sound Blaster',
    r'game & watch': 'Nintendo',
    r'ring fit': 'Nintendo',
    r'stealth ?cam': 'Stealth Cam',
    r'sig sauer': 'Sig Sauer',
    r'star wars': 'Star Wars',
    r'sirius-?xm': 'Siriusxm',
    r'on-?stage': 'On-Stage',
    r'pyle-pro': 'Pyle',
    r'boyalink': 'Boya',
    r'canoncl-': 'Canon',
    r'c2g': 'C2G',
}
MULTI_WORD_RE = [(re.compile(rf'^({pat})\b', re.I), brand) for pat, brand in MULTI_WORD_BRANDS.items()]
MARKS = re.compile(r'[​‌‍﻿®™©]|\((r|tm)\)', re.I)

# Descriptor words that start some titles in place of a brand ("Wireless
# Mouse", "Genuine Canon Ink", "64XL Ink Cartridges ... for HP"). When the
# first word is one of these, look for a known brand later in the title
# (before any "for / compatible with", so "for HP" never makes it HP);
# failing that the listing is Unbranded -- generic/white-label products that
# don't name a maker in the title.
GENERIC_FIRST = {
    'wireless', 'usb', 'usb-c', 'bluetooth', 'battery', 'batteries', 'laptop', 'gaming', 'power', 'case', 'ink',
    'desktop', 'waterproof', 'water', 'a', 'genuine', 'original', 'oem', 'bundle', 'replacement', 'rechargeable',
    'size', 'super', 'matte', 'aa', 'aaa', 'industrial', 'elite', 'lr44', 'cr', '3x', 'ups', 'compatible',
    'remanufactured', 'universal', 'portable', 'premium', 'mini', 'digital', 'smart', 'outdoor', 'all-new', 'kids', 'breadboard', 'breadboards',
}
# Tokens with digits are model codes (AX3000, B450M-HDV), except these brands
DIGIT_BRANDS = {'8bitdo', '3m', 'ds18', '1more'}
LENS_SHAPE = re.compile(r'^\d+(-\d+)?mm$', re.I)

def first_word(title):
    title = MARKS.sub('', str(title))
    # drop a leading bracketed tag, e.g. "[Older Version] SanDisk ..."
    title = re.sub(r'^\[[^\]]*\]\s*', '', title).strip()
    # "Made for Amazon" is a licensing program, not the maker ("Made for Amazon SanDisk ...")
    title = re.sub(r'^made for amazon,?\s*', '', title, flags=re.I)
    for pat, brand in MULTI_WORD_RE:
        if pat.match(title):
            return brand
    words = [w.strip(',').rstrip('-') for w in title.split()]
    for w in words[:5]:
        if w and not NON_BRAND_SHAPE.match(w):
            return w
    return words[0] if words else ''

def extract_brand(title, known):
    b = first_word(title)
    model_code = re.search(r'\d', b) and b.lower() not in DIGIT_BRANDS and b not in MULTI_WORD_BRANDS.values()
    if b.lower() not in GENERIC_FIRST and not LENS_SHAPE.match(b) and not model_code:
        return b
    head = COMPAT.split(MARKS.sub('', str(title)), maxsplit=1)[0]
    for w in re.split(r'[\s,]+', head)[1:]:
        w = w.strip(',:;()-').lower()
        if w in known:
            return w
    return 'Unbranded'

first = df['Title'].drop_duplicates().apply(first_word)
counts = first.str.lower().value_counts()
known = set(counts[counts >= 3].index) - GENERIC_FIRST

assert extract_brand('4Pc Size C Panasonic Batteries', known) == 'panasonic'
assert extract_brand('64XL Ink Cartridges Remanufactured for HP Ink 64', known) == 'Unbranded'
assert extract_brand('Genuine Canon PG-240XL', known) == 'canon'
assert extract_brand('2025 AX3000 Outdoor WiFi 6 Extender, WAVLINK Long-Range', known) == 'wavlink'
assert first_word('Intel® Core i9') == 'Intel'
assert first_word('​​PopSockets Grip') == 'PopSockets'
assert first_word('Midland- T51VP3') == 'Midland'
assert first_word('[Older Version] SanDisk 128GB') == 'SanDisk'
assert first_word('Texas Instruments TI-84') == 'Texas Instruments'
assert first_word('TI-84 Plus CE') == 'Texas Instruments'
assert first_word('2-Pack Duracell AA') == 'Duracell'
assert first_word('3M Privacy Filter') == '3M'

df['brand'] = df['Title'].apply(extract_brand, known=known).str.title()
# Product lines and spelling variants of one maker -> the maker. Separate brands
# that shoppers see on their own (Alienware, Beats, Xbox, Ring, Blink, eero,
# Soundcore, Tapo, HyperX, ...) stay separate, per the Amazon-brand rule.
SUB_BRANDS = {'Wd_Black': 'Western Digital', 'Wd': 'Western Digital', 'Hewlett-Packard': 'Hp',
              'Rog': 'Asus', 'Tuf': 'Asus', 'Legion': 'Lenovo', 'Predator': 'Acer',
              'Joy-Con': 'Nintendo', 'Oculus': 'Meta'}
df['brand'] = df['brand'].replace(SUB_BRANDS)
print((df.drop_duplicates('Title')['brand'] == 'Unbranded').sum(), 'products Unbranded')
df['brand'].value_counts().head(10)

40 products Unbranded


brand
Asurion     3067
Hp          1975
Sony         969
Dell         927
Lenovo       883
Samsung      869
Canon        792
Duracell     698
Ugreen       694
Apple        645
Name: count, dtype: int64

## Scope: electronics only

Drop protection plans (no price, no sales) and non-electronics (title-based rule; catches 110 of 114 hand-marked).

In [6]:
plans = df['category'] == 'Protection Plan'
assert not (df.loc[~plans, 'brand'] == 'Asurion').any()
print(f"protection plans: {plans.sum()} rows ({df.loc[plans, 'Title'].nunique()} titles), "
      f"in stock {df.loc[plans, 'in_stock'].mean():.0%}, with sales {df.loc[plans, 'bought_in_last_month'].notna().sum()}")
df = df[~plans].reset_index(drop=True)

non_elec = df['category'] == 'Non-Electronics'
stationery = non_elec & df['Title'].map(is_stationery)
(df.loc[stationery].drop_duplicates('Title')[['Title', 'brand', 'current_price']]
   .sort_values('Title').to_csv('../data/dropped_stationery.csv', index=False))
hand = set(pd.read_csv('../data/non_electronics_handmarked.csv')['Title'])
flagged = set(df.loc[non_elec, 'Title'])
print(f"non-electronics: {len(flagged)} products flagged ({df.loc[stationery, 'Title'].nunique()} plain stationery); "
      f"catches {len(hand & flagged)} of {len(hand)} hand-marked")
print('hand-marked but not flagged:', sorted(t[:45] for t in hand - flagged))

protection plans: 3068 rows (666 titles), in stock 0%, with sales 0


non-electronics: 171 products flagged (55 plain stationery); catches 110 of 114 hand-marked
hand-marked but not flagged: ["Burt's Bees Back To School Dorm Essentials, C", 'JansSport Pack BIG STUDENT GRAPHITE GREY', "Kipling Women's Seoul Small Tablet Backpack, ", 'Sprayground PACMAN GHOST GANG SHARMOUTH']


In [7]:
print(f"dropping {non_elec.sum()} listing rows ({df.loc[non_elec, 'Title'].nunique()} distinct products)")
df = df[~non_elec].reset_index(drop=True)

dropping 821 listing rows (171 distinct products)


## Reference labels and category model

Hand, gold and Sonnet labels replace the automatic label where they exist; a logistic regression (title embedding + rule label, out-of-fold) gives `category_auto`.

In [8]:
def to_current(title, label):
    """Re-apply the Revision 4 rules to a label made before them."""
    t = title.lower()
    if label == 'Phone & Tablet Accessories' and re.search(PHONES, t):
        return 'Phones & Tablets'
    if label in ('Phone & Tablet Accessories', 'Monitors & Displays') and re.search(WATCHES, t):
        return 'Smart Home & Wearables'
    if label == 'Computers & Laptops' and categorize(title) == 'Storage & Memory Cards':
        return 'Storage & Memory Cards'
    if label in CAR_FROM and CAR.search(title):
        return 'Car Electronics'
    if label in ('Cables & Charging', 'Other') and re.search(VIDEO, t):  # not Computers: laptops list a webcam
        return 'Cameras & Photography'
    if label == 'Audio' and re.search(VIDEO_SYSTEM, t):
        return 'Cameras & Photography'
    if label != 'Power Strips & Surge Protectors' and categorize(title) == 'Power Strips & Surge Protectors' and re.search(UPS, t):
        return 'Power Strips & Surge Protectors'  # UPS units (user's call, 2026-09-25)
    if label == 'Office & School Supplies' and is_stationery(title):
        return 'Non-Electronics'  # plain stationery (user's call, 2026-09-25)
    return label

user = pd.concat([pd.read_csv('../labeling/random_250_labels.csv'),
                  pd.read_csv('../labeling/sonnet_unsure_47_labels.csv')])
gold = pd.read_csv('../data/category_gold_sample.csv')
sonnet = pd.read_csv('../labeling/sonnet_labels.csv')
r250 = pd.read_csv('../labeling/random_250_labels.csv')
refs = pd.concat([
    user.assign(source='user', label=user['true_category'],
                eval_set=np.where(user['Title'].isin(r250['Title']) & (user['verdict'] != 'unsure'), 'random_250', '')),
    gold.assign(source='gold', label=gold['true_category'], eval_set='gold_' + gold['sample'].astype(str)),
    sonnet.assign(source=np.where(sonnet['confident'], 'sonnet_confident', 'sonnet_unsure'), label=sonnet['sonnet'], eval_set=''),
])[['Title', 'source', 'eval_set', 'label']]
refs['label_now'] = [to_current(t, l) for t, l in zip(refs['Title'], refs['label'])]
refs.to_csv('../data/category_reference_labels.csv', index=False)  # read by category_check.ipynb
print(f"{len(refs)} reference labels ({refs['Title'].nunique()} products); "
      f"{(refs['label'] != refs['label_now']).sum()} re-mapped to the current taxonomy")
assert set(refs['label_now']) <= set(df['category']) | {'Non-Electronics'}

priority = ['user', 'gold', 'sonnet_confident', 'sonnet_unsure']
ref = (refs.assign(p=refs['source'].map(priority.index)).sort_values('p')
           .drop_duplicates('Title').set_index('Title'))

1024 reference labels (1002 products); 61 re-mapped to the current taxonomy


In [9]:
import os
for k, v in {'HF_HUB_VERBOSITY': 'error', 'HF_HUB_DISABLE_PROGRESS_BARS': '1', 'TRANSFORMERS_VERBOSITY': 'error'}.items():
    os.environ.setdefault(k, v)  # quiet model-download chatter
from sentence_transformers import SentenceTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold

# category is a function of Title, so work on distinct titles
titles = df.drop_duplicates('Title').set_index('Title')['category']
head = titles.index.str.split(r'\b(?:for|compatible with|works with)\b|[,|(\[]', n=1, regex=True).str[0].str.strip()
head = pd.Series(head, index=titles.index).where(lambda s: s.str.len() >= 15, titles.index.to_series())

model = SentenceTransformer('all-MiniLM-L6-v2')
emb = model.encode(head.tolist(), batch_size=128, normalize_embeddings=True, show_progress_bar=False)
cats = np.array(sorted(titles.unique()))
rule = (titles.to_numpy(dtype=object)[:, None] == cats[None, :]).astype(float)
X = np.hstack([emb, 3 * rule])  # x3: rough scale match to the unit-length embedding, untuned

y = titles.index.to_series().map(ref['label_now']).fillna(titles).to_numpy(dtype=object)  # corrected labels
trainable = y != 'Non-Electronics'  # too few to learn; those products get dropped below anyway
pred = np.empty(len(titles), dtype=object)
for tr, te in KFold(5, shuffle=True, random_state=0).split(X):
    tr = tr[trainable[tr]]
    pred[te] = LogisticRegression(C=10, max_iter=3000).fit(X[tr], y[tr]).predict(X[te])

changed = pred != titles.to_numpy(dtype=object)
print(f'model changed the keyword label for {changed.sum()} of {len(titles)} products ({changed.mean():.1%})')
df['category_regex'] = df['category']
df['category_auto'] = df['Title'].map(pd.Series(pred, index=titles.index))
pd.DataFrame({'rule label': titles[changed].to_numpy(), 'model label': pred[changed]}).value_counts().head(10)  # most common changes

model changed the keyword label for 284 of 7971 products (3.6%)


rule label                  model label               
Other                       Cameras & Photography         31
                            Computers & Laptops           24
Computers & Laptops         Audio                         23
Other                       Smart Home & Wearables        20
                            Audio                         17
Phone & Tablet Accessories  Cables & Charging             14
Computers & Laptops         Cables & Charging             13
Monitors & Displays         Audio                         12
Other                       Storage & Memory Cards        10
                            Phone & Tablet Accessories     6
Name: count, dtype: int64

In [10]:
df['category'] = df['Title'].map(ref['label_now']).fillna(df['category_auto'])
df['category_source'] = df['Title'].map(ref['source']).fillna('pipeline')
per_title = df.drop_duplicates('Title')
changed = per_title['category'] != per_title['category_auto']
print(f"{changed.sum()} of {len(per_title)} distinct products relabelled, by source:")
print(per_title[changed]['category_source'].value_counts().to_string())

256 of 7971 distinct products relabelled, by source:
category_source
sonnet_confident    190
sonnet_unsure        39
user                 18
gold                  9


In [11]:
relabelled_out = df['category'] == 'Non-Electronics'
# stationery that reached Office via a reference label rather than the rules: add to the log
late = df.loc[relabelled_out & df['Title'].map(is_stationery)].drop_duplicates('Title')[['Title', 'brand', 'current_price']]
log = pd.concat([pd.read_csv('../data/dropped_stationery.csv'), late]).drop_duplicates('Title').sort_values('Title')
log.to_csv('../data/dropped_stationery.csv', index=False)
print(f'stationery log: {len(log)} products ({len(late)} caught at the reference-label step)')
print(f"dropping {relabelled_out.sum()} rows ({df.loc[relabelled_out, 'Title'].nunique()} products) relabelled Non-Electronics")
df = df[~relabelled_out].reset_index(drop=True)

stationery log: 58 products (3 caught at the reference-label step)
dropping 623 rows (25 products) relabelled Non-Electronics


## Sanity checks and save

Extremes are real products; missing ratings cluster on new, sponsored listings.

In [12]:
for col, f in [('current_price', 'nlargest'), ('current_price', 'nsmallest'), ('number_of_reviews', 'nlargest')]:
    print(f'{col} {f}:', getattr(df, f)(3, col).apply(lambda r: f"{r['Title'][:45]} ({r[col]:,.2f})", axis=1).tolist())
missing = df['rating'].isna()
print(f"rating missing: {missing.sum()} rows; no sales value {df.loc[missing, 'bought_in_last_month'].isna().mean():.0%} "
      f"vs {df.loc[~missing, 'bought_in_last_month'].isna().mean():.0%}, sponsored {df.loc[missing, 'is_sponsored'].mean():.0%} "
      f"vs {df.loc[~missing, 'is_sponsored'].mean():.0%}")
df.to_csv('../data/amazon_sales_data_cleaned.csv', index=False)
print('saved listing file:', df.shape)

current_price nlargest: ['Logitech Rally bar and tap IP Bundle, 1920 Pi (4,699.00)', 'ASUS ROG Strix SCAR 18 (2025) Gaming Laptop,  (4,399.00)', 'Canon EOS R3 Mirrorless Camera (Body Only), F (4,399.00)']
current_price nsmallest: ['Woods Light Duty Extension Cord Wrap, Holds u (2.97)', 'Casio HS-8VA Mini 6-Function Calculator | Lar (3.59)', 'Duracell 376/377 Silver Oxide Button Battery, (3.65)']
number_of_reviews nlargest: ['Amazon Basics 48-Pack AA Alkaline High-Perfor (865,598.00)', 'SanDisk 32GB Ultra microSDHC UHS-I Memory Car (645,418.00)', '[Older Version] SanDisk 32GB 2-Pack Ultra Mic (645,416.00)']
rating missing: 747 rows; no sales value 26% vs 7%, sponsored 45% vs 15%
saved listing file: (37201, 13)


## Product file

One row per title (repeat scrapes dominate rows); `product_family` groups near-identical variants.

In [13]:
products = (
    df.sort_values('collected_at')
      .drop_duplicates(subset='Title', keep='last')
      .reset_index(drop=True)
)
print(f'{len(df)} listing-observations -> {len(products)} distinct products (by Title)')

# product_family: near-identical listings of the same brand (colour / capacity /
# pack-size variants, "Renewed" vs "Renewed Premium", re-listed titles).
# Same brand + title-embedding cosine >= 0.95, joined transitively.
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
E = model.encode(products['Title'].tolist(), batch_size=128, normalize_embeddings=True, show_progress_bar=False)
b = products['brand'].to_numpy(dtype=object)
_, products['product_family'] = connected_components(csr_matrix((E @ E.T >= 0.95) & (b[:, None] == b[None, :])), directed=False)
fam_size = products['product_family'].map(products['product_family'].value_counts())
print(f"{(fam_size > 1).sum()} products in {products.loc[fam_size > 1, 'product_family'].nunique()} multi-listing families; largest {fam_size.max()}")
products.to_csv('../data/amazon_products_deduped.csv', index=False)
products['category'].value_counts()

37201 listing-observations -> 7946 distinct products (by Title)


1606 products in 594 multi-listing families; largest 19


category
Audio                              1205
Computers & Laptops                1152
Printers & Ink                      910
Cameras & Photography               727
Phone & Tablet Accessories          417
Cables & Charging                   378
Monitors & Displays                 332
Storage & Memory Cards              331
Batteries                           321
Networking                          315
Keyboards & Mice                    299
Office & School Supplies            266
Smart Home & Wearables              227
TV & Home Entertainment             205
Phones & Tablets                    186
Gaming                              155
Car Electronics                     146
Power Strips & Surge Protectors     139
Bags & Cases                        122
Marine & GPS                         52
Optics                               48
Other                                13
Name: count, dtype: int64